# 01 - Extração: MongoDB → MinIO (JSON)

Extrai **todas as coleções** do database `VendasArCondicionado` do MongoDB e exporta como JSON para o bucket `landing-zone` no MinIO.

**Pré-requisitos:** Docker Compose rodando, Notebook `00` executado.

## 1. Configuração

In [11]:
import os
import json
import boto3
from pymongo import MongoClient
from botocore.client import Config
from dotenv import load_dotenv

load_dotenv(override=True)

DB_HOST     = os.getenv('DB_HOST', 'localhost')
DB_PORT     = os.getenv('DB_PORT', '27017')
DB_USER     = os.getenv('DB_USER', 'admin')
DB_PASSWORD = os.getenv('DB_PASSWORD', '1234')
DB_NAME     = 'VendasArCondicionado'

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT', 'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET', 'landing-zone')

print(f'MongoDB: {DB_HOST}:{DB_PORT}/{DB_NAME}')
print(f'MinIO: {MINIO_ENDPOINT} | Bucket: {LANDING_BUCKET}')

MongoDB: localhost:27017/VendasArCondicionado
MinIO: http://localhost:9020 | Bucket: landing-zone


## 2. Conexão com MongoDB

In [12]:
uri = f"mongodb://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/"
client = MongoClient(uri)
db = client[DB_NAME]

print(f'Conectado ao MongoDB [{DB_NAME}] com sucesso!')

Conectado ao MongoDB [VendasArCondicionado] com sucesso!


## 3. Listar Coleções

In [13]:
colecoes = db.list_collection_names()
colecoes.sort()

print(f'{len(colecoes)} coleções encontradas:')
for i, col in enumerate(colecoes, 1):
    count = db[col].count_documents({})
    print(f'  {i:2d}. {col:<20} ({count:>6} documentos)')

4 coleções encontradas:
   1. ar_condicionado      (     3 documentos)
   2. clientes             (     2 documentos)
   3. vendas               (     2 documentos)
   4. vendedores           (     2 documentos)


## 4. Conectar e Criar Bucket no MinIO

In [14]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] já existe.')
except:
    s3_client.create_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] criado com sucesso!')

print('Buckets disponíveis:', [b['Name'] for b in s3_client.list_buckets()['Buckets']])

Bucket [landing-zone] já existe.
Buckets disponíveis: ['landing-zone']


## 5. Extrair Coleções → JSON no MinIO

In [15]:
print(f'\nExtraindo {len(colecoes)} coleções...\n')
resultados = []

for colecao_nome in colecoes:
    # 1. Buscar todos os documentos da coleção
    cursor = db[colecao_nome].find()
    dados = list(cursor)
    
    if not dados:
        print(f'  ⚠️ {colecao_nome} está vazia. Pulando...')
        continue

    # 2. Converter para JSON
    # O parâmetro default=str transforma os 'ObjectIds' do Mongo em string, evitando erros
    json_str = json.dumps(dados, default=str, ensure_ascii=False)
    json_bytes = json_str.encode('utf-8')
    
    # 3. Definir o nome do arquivo no MinIO (agora .json)
    s3_key = f'{colecao_nome}.json'

    # 4. Fazer o Upload para o MinIO
    s3_client.put_object(
        Bucket=LANDING_BUCKET, 
        Key=s3_key,
        Body=json_bytes, 
        ContentType='application/json'
    )

    size_kb = len(json_bytes) / 1024
    resultados.append({
        'colecao': colecao_nome, 
        'documentos': len(dados), 
        'tamanho_kb': round(size_kb, 1)
    })
    
    print(f'  ✅ {colecao_nome}.json -> s3://{LANDING_BUCKET}/{s3_key} ({len(dados)} docs, {size_kb:.1f} KB)')

print(f'\nExtração concluída! Arquivos exportados.')


Extraindo 4 coleções...

  ✅ ar_condicionado.json -> s3://landing-zone/ar_condicionado.json (3 docs, 0.5 KB)
  ✅ clientes.json -> s3://landing-zone/clientes.json (2 docs, 0.2 KB)
  ✅ vendas.json -> s3://landing-zone/vendas.json (2 docs, 0.3 KB)
  ✅ vendedores.json -> s3://landing-zone/vendedores.json (2 docs, 0.2 KB)

Extração concluída! Arquivos exportados.


## 6. Validação no MinIO

In [16]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
print(f'Arquivos no bucket [{LANDING_BUCKET}]:\n')

if 'Contents' in response:
    for obj in response['Contents']:
        print(f'  {obj["Key"]:<25} {obj["Size"]/1024:>8.1f} KB')
else:
    print("  Nenhum arquivo encontrado no bucket.")

Arquivos no bucket [landing-zone]:

  ar_condicionado.json           0.5 KB
  clientes.json                  0.2 KB
  vendas.json                    0.3 KB
  vendedores.json                0.2 KB


In [17]:
client.close()
print('\nConexão MongoDB encerrada.')


Conexão MongoDB encerrada.
